<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/Evaluate/model24750.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# pickle dosyasını yükle
with open("/content/drive/MyDrive/datasets/merged_europarl_informal.pkl", "rb") as f:
    df = pd.read_pickle(f)

# cinsiyet dağılımı
print(df["gender"].value_counts())
print("\nOranlar (%):")
print(df["gender"].value_counts(normalize=True) * 100)

gender
female    591027
male      591027
Name: count, dtype: int64

Oranlar (%):
gender
female    50.0
male      50.0
Name: proportion, dtype: float64


In [ ]:
# gerekli kütüphaneler
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader

# veriyi yükle
dataset_path = "/content/drive/MyDrive/datasets/merged_europarl_informal.pkl"
with open(dataset_path, "rb") as f:
    df = pd.read_pickle(f)

# gpu kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model ve tokenizer yükle
model_path = "/content/drive/MyDrive/models/gp_model_24750"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

# veri seti sınıfı
class GenderDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# etiketleri sayıya çevir
label_map = {"female": 0, "male": 1}
df = df[df["gender"].isin(label_map)]
df["label"] = df["gender"].map(label_map)

# dataset ve dataloader
dataset = GenderDataset(df["text"].tolist(), df["label"].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

# tahminleri yap
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# metrikleri yazdır
report = classification_report(all_labels, all_preds, target_names=["female", "male"])
print(report)


              precision    recall  f1-score   support

      female       0.63      0.48      0.54    591027
        male       0.58      0.72      0.64    591027

    accuracy                           0.60   1182054
   macro avg       0.61      0.60      0.59   1182054
weighted avg       0.61      0.60      0.59   1182054



In [ ]:
import requests

def send_telegram_message(message):
    token = "7791020893:AAGIXZbLRG6YVNXaNhRkCNiQDUV2-jXsDJY"
    chat_id = 7689600055
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    data = {"chat_id": chat_id, "text": message}
    requests.post(url, data=data)

send_telegram_message("Colab çalışman tamamlandı 🎉")